# Week 50, Day 4: Fitting the diffusion model by maximum likelihood

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w050/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The density and its checks

Write fpt_lower(t, v, a, w, K=60) from the series. Integrate it numerically on a fine grid for v = 1.2, a = 1.5, w = 0.5 and compare with the analytic error probability 1 / (1 + exp(v a)) for an unbiased start and with simulated data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(2009)

def fpt_lower(t, v, a, w, K=60):
    ...

def simulate(v, a, t0, n=1500, dt=0.001):
    x = np.full(n, a / 2); done = np.zeros(n, bool); ch = np.zeros(n, int); rt = np.zeros(n); k = 0
    while not done.all() and k < 20000:
        idx = ~done; x[idx] += v * dt + np.sqrt(dt) * r.normal(size=idx.sum()); k += 1
        up, lo = idx & (x >= a), idx & (x <= 0)
        ch[up] = 1; rt[up | lo] = k * dt + t0; done |= up | lo
    return ch, rt

ch, rt = simulate(1.2, 1.5, 0.3)
tt = np.arange(0.0005, 6, 0.001)
area_lower = fpt_lower(tt, 1.2, 1.5, 0.5).sum() * 0.001
p_lower_theory = 1 / (1 + np.exp(1.2 * 1.5))
print(f"integrated lower density {area_lower:.4f}; analytic {p_lower_theory:.4f}; simulated error rate {1 - ch.mean():.4f}")
plt.hist(rt[ch == 1] - 0.3, 60, density=True, alpha=0.4, label="simulated correct")
plt.plot(tt, fpt_lower(tt, -1.2, 1.5, 0.5) / (1 - area_lower), label="density (upper, normalised)"); plt.xlim(0, 3); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_area():
    "the lower-bound density integrates to the analytic probability within 0.002"
    assert abs(area_lower - p_lower_theory) < 0.002
def test_sim():
    "it matches the simulated error rate within 0.03"
    assert abs(area_lower - (1 - ch.mean())) < 0.03

test_area()
test_sim()
print("All checks passed")

### Exercise: Recovering v, a and t0

Write density(t, choice, v, a, t0): the upper-bound density for choice 1 and the lower for choice 0, evaluated at t - t0, with a small floor for times at or below t0. Minimise the negative log-likelihood with Nelder-Mead, returning a large value when a or t0 are out of range.

In [ ]:
def density(t, choice, v, a, t0, w=0.5):
    dt_ = t - t0; out = np.full(len(t), 1e-12); ok = dt_ > 1e-4
    ...
    return np.maximum(out, 1e-12)

def nll(p):
    v, a, t0 = p
    if a <= 0.2 or not (0 < t0 < rt.min()):
        return 1e10
    return -np.sum(np.log(density(rt, ch, v, a, t0)))

fit = minimize(nll, [0.5, 1.0, 0.1], method="Nelder-Mead", options={"maxiter": 2000, "xatol": 1e-4, "fatol": 1e-4})
print(f"fitted v = {fit.x[0]:.3f}, a = {fit.x[1]:.3f}, t0 = {fit.x[2]:.3f} (true 1.2, 1.5, 0.3); NLL {fit.fun:.1f}")

In [ ]:
import numpy as np
def test_recover():
    "fitted drift within 0.15, boundary within 0.1 and non-decision time within 0.02 s"
    v_, a_, t_ = fit.x
    assert abs(v_ - 1.2) < 0.15 and abs(a_ - 1.5) < 0.1 and abs(t_ - 0.3) < 0.02

test_recover()
print("All checks passed")

### Exercise: Predicted against observed quantiles

From the fitted parameters, compute the cumulative distribution of correct response times on a fine grid (cumulative sum of the upper density, normalised by its total), invert it at 0.1, 0.3, 0.5, 0.7 and 0.9, add t0, and compare with the data's quantiles.

In [ ]:
v_, a_, t_ = fit.x
grid = np.arange(0.0005, 6, 0.001)
dens = fpt_lower(grid, -v_, a_, 0.5)
cdf = ...
probs = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
q_pred = ...
q_data = np.quantile(rt[ch == 1], probs)
print("quantiles  data:", np.round(q_data, 3)); print("quantiles model:", np.round(q_pred, 3))
plt.plot(q_data, probs, "o", label="data"); plt.plot(q_pred, probs, "x-", label="model"); plt.xlabel("response time (s)"); plt.ylabel("cumulative probability"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_quantiles():
    "predicted 10th to 90th percentiles of correct RTs are within 40 ms of the data"
    assert np.abs(q_pred - q_data).max() < 0.04

test_quantiles()
print("All checks passed")